<a href="https://colab.research.google.com/github/JorgeZorrilla/Crash-GeoNN/blob/main/TrainingModel_iteration_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TODO:
- Meter velocidades
- Meter aceleraciones
- Diferenciar los distintos solidos
- Meter features estaticos.
- Meter los del validation a k steps para tener mejores predicciones en el futuro
- Probar lo del CLAMP_BC_IN_ROLLOUT

## Configuration

In [4]:
# Configuration parameters
SEED_NUMBER = 42
MIN_T = 5
STEP = 2 # To select a smaller number of attributes from the database
LAM_BC = 1e-2
LAM_SMOOTH = 1e-3
CLAMP_BC_IN_ROLLOUT = False
PATIENCE = 15
MAX_EPOCHS = 200
N_LAYERS= 3
HIDDEN= 128


## Install dependencies

In [5]:
# Colab setup: install PyTorch Geometric wheels matching your Torch/CUDA
import torch, sys, os, platform, subprocess, textwrap
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda)

# This magic line pulls the right wheels for your torch+cuda combo
torch_ver = torch.__version__.split('+')[0]
cuda_tag = (torch.version.cuda or 'cpu').replace('.', '')
index_url = f"https://data.pyg.org/whl/torch-{torch_ver}%2B{cuda_tag}.html"

!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv torch_geometric \
  -f https://data.pyg.org/whl/torch-2.8.0+cu126.html

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)



Torch: 2.8.0+cu126 | CUDA: 12.6
Device: cuda


Mount drive

In [6]:
from google.colab import drive
drive.mount('/content/drive')  # autoriza y usa rutas como '/content/drive/MyDrive/...'


Mounted at /content/drive


Import dependencies

In [7]:
import os, math, random, numpy as np, time
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from typing import Dict, List, Tuple, Sequence
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GraphSAGE
from tqdm.auto import tqdm

Utilities

In [8]:
def set_seed(seed: int = 42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    try: torch.set_float32_matmul_precision("high")
    except: pass

def worker_init_fn(worker_id):
    seed = torch.initial_seed() % 2**31
    np.random.seed(seed + worker_id); random.seed(seed + worker_id)

def check_sim(steps: List[Data], sid: int, max_print_edges=5):
    assert isinstance(steps, (list, tuple)) and len(steps) >= 1, f"[sim {sid}] bad list"
    N = steps[0].x.shape[0]
    E = steps[0].edge_index.shape[1]
    pos0 = getattr(steps[0], 'pos0', None)
    edge_index0 = steps[0].edge_index
    issues = []
    for t, g in enumerate(steps):
        if not isinstance(g, Data): issues.append(f"step {t} not Data"); continue
        if g.x.dim()!=2 or g.y.dim()!=2: issues.append(f"step {t} x/y dim !=2")
        if g.x.shape[0]!=N or g.y.shape[0]!=N: issues.append(f"step {t} N mismatch")
        if g.edge_index.shape[0]!=2 or g.edge_index.shape[1]!=E: issues.append(f"step {t} ei shape mismatch")
        if not torch.equal(g.edge_index, edge_index0): issues.append(f"step {t} ei differs")
        if int(g.edge_index.max()) >= N: issues.append(f"step {t} ei out of range")
        if not torch.isfinite(g.x).all() or not torch.isfinite(g.y).all(): issues.append(f"step {t} NaN/Inf in x/y")
        if hasattr(g, "edge_attr"):
            if not torch.isfinite(g.edge_attr).all(): issues.append(f"step {t} NaN/Inf in edge_attr")
    ei = edge_index0.t().tolist()
    undirected = all(([j,i] in ei) for i,j in ei[:max_print_edges])
    unique_pairs = set(tuple(sorted(e)) for e in ei)
    dup = (len(unique_pairs) * 2 != len(ei))
    print(f"[sim {sid}] N={N} E={E} undirected? {undirected} duplicates? {dup}")
    if issues: print("  Issues:", "; ".join(issues))

def transform_edge_attr(edge_attr: torch.Tensor, edge_scaler):
    if edge_attr is None or edge_scaler is None:
        return edge_attr
    em, es = edge_scaler
    return (edge_attr - em) / es

def drop_features_db(db: List[List[Data]], drop_idx_x: List[int], drop_idx_y: List[int]):
    """
    Elimina atributos (columnas) de x (y opcionalmente de y) en TODA la base de datos.

    Args:
        db: List[List[Data]]  -> base de datos completa
        drop_idx: lista de índices de columnas a eliminar
    """
    if not drop_idx_x and not drop_idx_y:
        return db  # nada que hacer

    drop_idx_x = sorted(set(drop_idx_x))
    drop_idx_y = sorted(set(drop_idx_y))

    for sim in db:
        for g in sim:
            # --- X ---
            if hasattr(g, "x") and g.x is not None:
                keep_x = [i for i in range(g.x.size(1)) if i not in drop_idx_x]
                g.x = g.x[:, keep_x]

            # --- Y (opcional) ---
            if hasattr(g, "y") and g.y is not None:
              keep_y = [i for i in range(g.y.size(1)) if i not in drop_idx_y]
              g.y = g.y[:, keep_y]


    return db



def tensor3d_to_dfs(arr3d, feature_names=None):
    """
    arr3d: (T, N, C) en numpy o torch
    feature_names: lista de nombres de longitud C (opcional)
    """
    # -> numpy
    if isinstance(arr3d, torch.Tensor):
        A = arr3d.detach().cpu().numpy()
    else:
        A = np.asarray(arr3d)
    T, N, C = A.shape

    # Nombres de columnas
    cols = feature_names if feature_names is not None else [f"f{i}" for i in range(C)]

    # ---- Wide: index=(t,node), columns=features ----
    idx = pd.MultiIndex.from_product([range(T), range(N)], names=["t", "node"])
    df_wide = pd.DataFrame(A.reshape(T*N, C), index=idx, columns=cols)

    # ---- Long: tidy ----
    t_idx   = np.repeat(np.arange(T), N*C)
    node_idx= np.tile(np.repeat(np.arange(N), C), T)
    feat_idx= np.tile(np.arange(C), T*N)
    feat    = np.array(cols)[feat_idx]
    values  = A.reshape(-1)
    df_long = pd.DataFrame({"t": t_idx, "node": node_idx, "feature": feat, "value": values})

    return df_long, df_wide

In [9]:
set_seed(SEED_NUMBER)

Load Database

In [10]:
def load_database(path_pt: str) -> List[List[Data]]:
    # print("Loading DB from:", path_pt)
    db = torch.load(path_pt, map_location="cpu", weights_only=False)
    return db

def load_database_dir(dir_path: str, step = 1) -> List[List[Data]]:
    print("Loading DB from:", dir_path)
    db = []
    graphs = os.listdir(dir_path)
    if graphs:
      print(f"Found {len(graphs)} graphs")
      for i in tqdm(range(0, len(graphs), step)):
        path = os.path.join(dir_path, graphs[i])
        db.append(load_database(path))
      for sid, steps in enumerate(db[:5]): check_sim(steps, sid)
      lens = [len(s) for s in db]
    return db
# def load_database_dir(path_pt: str, step = 1) -> List[List[Data]]:
#     print("Loading DB from:", path_pt)
#     db = []
#     graphs = os.listdir(path_pt)
#     if graphs:
#       print(f"Found {len(graphs)} graphs")
#       for i in range(0, len(graphs), step):
#         print(f"Loading graph {graphs[i]}")
#         path = os.path.join(path_pt, graphs[i])
#         db.append(torch.load(path, map_location="cpu", weights_only=False))
#       assert isinstance(db, (list, tuple)) and all(isinstance(sim, (list, tuple)) for sim in db)
#       for sid, steps in enumerate(db[:5]): check_sim(steps, sid)
#       lens = [len(s) for s in db]
#       print(f"T-1 per sim (min/mean/max): {min(lens)}/{sum(lens)/len(lens):.1f}/{max(lens)}")
#       print(f"Loaded ${len(db)} graphs!")
#     return db

def split_simulations(all_sim_ids, train_ratio=0.7, val_ratio=0.15, seed=42):
    rng = np.random.default_rng(seed); ids = np.array(all_sim_ids); rng.shuffle(ids)
    n = len(ids); n_tr = int(n*train_ratio); n_va = int(n*val_ratio)
    return ids[:n_tr].tolist(), ids[n_tr:n_tr+n_va].tolist(), ids[n_tr+n_va:].tolist()

def build_split_from_db(db: List[List[Data]], sim_ids: List[int], min_time_step: int = 0):
    graphs, sim_static = [], {}
    for sid in sim_ids:
        steps_all = db[sid]
        assert len(steps_all) >= 1, f"Simulation {sid} empty."

        # Si no hay suficientes pasos, saltamos la simulación
        if len(steps_all) <= min_time_step:
            print(f"[WARN] sim {sid} skipped: len(steps)={len(steps_all)} <= min_t={min_time_step}")
            continue

        # Filtrado por timestep
        steps = steps_all[min_time_step:]                        # Data_t(min_time_step) .. Data_t(T-2)
        edge_index = steps[0].edge_index
        pos0 = getattr(steps[0], 'pos0', None)
        simulation_id = getattr(steps[0], 'simulation_id', None)
        bc_mask = getattr(steps[0], 'bc_mask', None)
        rigid_mask = getattr(steps[0], 'rigid_mask', None)
        timestep_index = getattr(steps[0], 't_idx', None)
        # fixed_idx = getattr(steps[0], 'fixed_idx', None)
        edge_attr = getattr(steps[0], 'edge_attr', None)

        # K = len(steps) = (T-1 - min_time_step)
        # Estados efectivos: ΔX_{min_time_step} .. ΔX_T  -> T_eff = K + 1
        T_eff = len(steps) + 1

        # Ground-truth a partir de min_time_step: ΔX_{min_time_step+1 .. T}
        y_real = torch.stack([d.y for d in steps], dim=0)  # (T_eff-1, N, N_features)

        # Estado inicial para rollout: ΔX_{min_t} (ojo: sin normalizar)
        x0 = steps_all[min_time_step].x.detach().clone()

        # Añadimos los Data filtrados al conjunto de entrenamiento/val/test
        graphs.extend(steps)

        sim_static[sid] = {
            'simulation_id' : simulation_id,
            'bc_mask' : bc_mask,
            'rigid_mask' : rigid_mask,
            'timestep_index' : timestep_index,
            # 'fixed_idx' : fixed_idx,
            'edge_index': edge_index,
            'edge_attr' : edge_attr,     # OJO: aún sin escalar aquí
            'pos0': pos0,
            'T_eff': T_eff, # Number of effective timesteps
            'y_real': y_real,
            'x0': x0           # punto de partida del rollout
        }

    return graphs, sim_static

In [11]:
# BBDD parameters
INPUT_DIR="/content/drive/MyDrive/CrashGeoNN/graphs_iteration_3/"
# Attributes [Delta_x, Delta_y, Delta_z, V_x, V_y, V_z, A_z, A_y, A_z, bc_mask, rigid_mask]
# N_ FEATURES = 11
X_DISCARD_INDEX = [3,4,5,6,7,8] # We start only with the coords and the static masks
X_DYNAMIC_INDEX = [0,1,2]
X_STATIC_INDEX = [3, 4] # After removing the previous index bc_mask and rigid_mask remain
Y_DISCARD_INDEX = [3,4,5,6,7,8] # We start only with the coords and the static masks
Y_DYNAMIC_INDEX = [0,1,2]
Y_STATIC_INDEX = [] # No static attributes here

In [12]:
# === Cambia esta ruta a tu .pt (Drive o local) ===
DB_PATH = INPUT_DIR  # p.ej.: "/content/drive/MyDrive/Crash-GeoNN/GRAPHS.pt"
simulations = load_database_dir(DB_PATH, STEP)


Loading DB from: /content/drive/MyDrive/CrashGeoNN/graphs_iteration_3/
Found 93 graphs


  0%|          | 0/47 [00:00<?, ?it/s]

[sim 0] N=3544 E=28108 undirected? True duplicates? False
[sim 1] N=3499 E=27772 undirected? True duplicates? False
[sim 2] N=3546 E=28152 undirected? True duplicates? False
[sim 3] N=3555 E=28196 undirected? True duplicates? False
[sim 4] N=3550 E=28168 undirected? True duplicates? False


In [13]:
print(f"Number of features before: {simulations[0][0].x.shape[1]}")
simulations = drop_features_db(simulations,X_DISCARD_INDEX, Y_DISCARD_INDEX)
INPUT_FEATURES = simulations[0][0].x.shape[1]
OUTPUT_FEATURES = simulations[0][0].y.shape[1]
print(f"Final number of INPUT features: {INPUT_FEATURES}. Dynamic: {len(X_DYNAMIC_INDEX)}. Static: {len(X_STATIC_INDEX)}")
assert INPUT_FEATURES == (len(X_DYNAMIC_INDEX) + len(X_STATIC_INDEX))
print(f"Final number of OUTPUT features: {OUTPUT_FEATURES}. Dynamic: {len(Y_DYNAMIC_INDEX)}. Static: {len(Y_STATIC_INDEX)}")
assert OUTPUT_FEATURES == (len(Y_DYNAMIC_INDEX) + len(Y_STATIC_INDEX))



all_ids = list(range(len(simulations)))
train_ids, val_ids, test_ids = split_simulations(all_ids, train_ratio=0.7, val_ratio=0.15, seed=42)

Number of features before: 11
Final number of INPUT features: 5. Dynamic: 3. Static: 2
Final number of OUTPUT features: 3. Dynamic: 3. Static: 0


In [14]:
print("Building splits...")
train_graphs, train_static = build_split_from_db(simulations, train_ids, min_time_step=MIN_T)
val_graphs,   val_static   = build_split_from_db(simulations, val_ids, min_time_step=MIN_T)
test_graphs,  test_static  = build_split_from_db(simulations, test_ids, min_time_step=MIN_T)


Building splits...


## Normalization

In [15]:
def fit_scaler(graphs: List[Data],
               affected_index_x: Sequence[int],
               affected_index_y: Sequence[int]):

    # Complete dimensions
    len_x = graphs[0].x.size(-1)
    len_y = graphs[0].y.size(-1)

    # Scaler X: only the affected columns
    Xc = torch.cat([g.x[:, affected_index_x] for g in graphs], dim=0)
    xm_c = Xc.mean(0, keepdim=True)
    xs_c = Xc.std(0, keepdim=True).clamp_min(1e-8)

    # We include average mu = 0 and std = 1 to the non affected attributes
    xm = torch.zeros(1, len_x, dtype=xm_c.dtype, device=xm_c.device)
    xs = torch.ones(1,  len_x, dtype=xs_c.dtype, device=xs_c.device)
    xm[:, affected_index_x] = xm_c # Modify affected attributes with average
    xs[:, affected_index_x] = xs_c # Modify affected attributes with average

    # Scaler Y: only the affected columns
    Yc = torch.cat([g.y[:, affected_index_y] for g in graphs], dim=0)
    ym_c = Yc.mean(0, keepdim=True)
    ys_c = Yc.std(0, keepdim=True).clamp_min(1e-8)

    # We include average mu = 0 and std = 1 to the non affected attributes
    ym = torch.zeros(1, len_y, dtype=ym_c.dtype, device=ym_c.device)
    ys = torch.ones(1,  len_y, dtype=ys_c.dtype, device=ys_c.device)
    ym[:, affected_index_y] = ym_c # Modify affected attributes with average
    ys[:, affected_index_y] = ys_c # Modify affected attributes with average

    return (xm, xs), (ym, ys)


def apply_scaler(graphs: List[Data], x_scaler, y_scaler):
    xm, xs = x_scaler
    ym, ys = y_scaler
    for g in graphs:
        dev = g.x.device
        g.x = (g.x - xm.to(dev)) / xs.to(dev)
        g.y = (g.y - ym.to(dev)) / ys.to(dev)

def fit_edge_attr_scaler(graphs: List[Data]):
    E_list = [g.edge_attr for g in graphs if hasattr(g, "edge_attr") and g.edge_attr is not None]
    if not E_list: return None
    E = torch.cat(E_list, dim=0)
    em, es = E.mean(0, keepdim=True), E.std(0, keepdim=True).clamp_min(1e-8)
    return (em, es)

def apply_edge_attr_scaler(graphs: List[Data], scaler):
    if scaler is None: return
    em, es = scaler
    for g in graphs:
        if hasattr(g, "edge_attr") and g.edge_attr is not None:
            g.edge_attr = (g.edge_attr - em) / es

In [16]:
print("Fitting scalers on TRAIN...")

x_scaler, y_scaler = fit_scaler(train_graphs, X_DYNAMIC_INDEX, Y_DYNAMIC_INDEX)
apply_scaler(train_graphs, x_scaler, y_scaler)
apply_scaler(val_graphs,   x_scaler, y_scaler)
apply_scaler(test_graphs,  x_scaler, y_scaler)

edge_scaler = fit_edge_attr_scaler(train_graphs)
apply_edge_attr_scaler(train_graphs, edge_scaler)
apply_edge_attr_scaler(val_graphs,   edge_scaler)
apply_edge_attr_scaler(test_graphs,  edge_scaler)

Fitting scalers on TRAIN...


Models

In [17]:
from torch_geometric.nn import GINEConv, BatchNorm, LayerNorm, GraphNorm

class ImpactGNN(nn.Module):
    def __init__(self, in_ch=3, hidden=128, out_ch=3, layers=3):
        super().__init__()
        self.gnn = GraphSAGE(in_channels=in_ch, hidden_channels=hidden, num_layers=layers)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, out_ch))
    def forward(self, x, edge_index, edge_attr=None):
        h = self.gnn(x, edge_index)
        return self.head(h)

class ImpactGNN_Edge(nn.Module):
    def __init__(self, in_ch=3, edge_attr_dim=4, hidden=128, out_ch=3, layers=3, dropout=0.1):
        super().__init__()
        convs, norms = [], []
        for l in range(layers):
            mlp = nn.Sequential(
                nn.Linear(hidden if l>0 else in_ch, hidden),
                nn.ReLU(),
                nn.Linear(hidden, hidden)
            )
            convs.append(GINEConv(mlp, edge_dim=edge_attr_dim))
            norms.append(BatchNorm(hidden))
        self.convs = nn.ModuleList(convs)
        self.norms = nn.ModuleList(norms)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, out_ch))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, edge_attr):
        h = x
        for conv, bn in zip(self.convs, self.norms):
            h = conv(h, edge_index, edge_attr)
            h = bn(h); h = F.relu(h); h = self.dropout(h)
        return self.head(h)

In [18]:
print("Creating loaders...")
loader_kwargs = dict(batch_size=16, shuffle=True, pin_memory=(device=='cuda'),
                      num_workers=0, worker_init_fn=worker_init_fn, persistent_workers=False)
train_loader = DataLoader(train_graphs, **loader_kwargs)
val_loader   = DataLoader(val_graphs,   **{**loader_kwargs, "shuffle": False})
test_loader  = DataLoader(test_graphs,  **{**loader_kwargs, "shuffle": False})

Creating loaders...


Training

In [19]:
def smooth_edge_penalty(pred, target, edge_index, lam=1e-3,
                        bc_mask=None, solid_id=None):
    """Match the gradient of the edge pred vs target, but ignores
    the edges that have nodes in the BC or belong to different solids."""
    src, dst = edge_index
    diff = (pred[src] - pred[dst]) - (target[src] - target[dst])  # [E, C]

    if bc_mask is not None:
        free_edge = ((bc_mask[src] == 0) & (bc_mask[dst] == 0)).unsqueeze(-1)  # [E,1]
        diff = diff * free_edge

    if solid_id is not None:
        same_solid = (solid_id[src] == solid_id[dst]).unsqueeze(-1)  # [E,1]
        diff = diff * same_solid

    return lam * diff.pow(2).mean()

def masked_mse(pred, target, mask_free):
    # mask_free: True en nodos libres
    if mask_free is None: return F.mse_loss(pred, target)
    pred_f, tgt_f = pred[mask_free], target[mask_free]
    return F.mse_loss(pred_f, tgt_f)

def loss_bc_zero_disp(pred_norm, dynamic_features, bc_mask,
                      y_scaler, lam_bc: float = 1e-3):
    """
    Penaliza desplazamiento != 0 EN ESPACIO FÍSICO en nodos fijos (bc_mask=True).
    pred_norm: y_hat normalizado
    y_scaler: (mean, std) usados para normalizar y. Si None, asumimos ya absoluto.
    """
    # TODO: Include custom weights for each features(some could be noisier)
    if lam_bc <= 0 or bc_mask is None or bc_mask.sum() == 0:
        return pred_norm.new_tensor(0.0)
    if y_scaler is None:
        pred_phys = pred_norm
    else:
        ym, ys = y_scaler
        pred_phys = pred_norm * ys.to(pred_norm) + ym.to(pred_norm)
    dynamic_features = pred_phys[:, dynamic_features]
    return lam_bc * (dynamic_features[bc_mask] ** 2).mean()

def train_epoch(model, loader, opt, dyn_features_idx_y, y_scaler, device='cuda', lam_smooth=1e-3, scaler=None, max_grad_norm=1.0):
    model.train()
    total, nodes = 0.0, 0
    for g in tqdm(loader, leave=False):
        g = g.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(scaler is not None)):
            pred = model(g.x, g.edge_index, getattr(g, 'edge_attr', None))
            bc_mask = getattr(g, 'bc_mask', None)
            if bc_mask is not None:
              bc_mask = bc_mask.bool()
              mask_free = ~bc_mask
            else:
              mask_free = None
            rigid_mask = getattr(g, 'rigid_mask', None)
            loss = smooth_edge_penalty(pred, g.y, g.edge_index, lam_smooth, bc_mask, rigid_mask)
            loss += masked_mse(pred, g.y, mask_free)
            loss += loss_bc_zero_disp(pred, dyn_features_idx_y, bc_mask, y_scaler, LAM_BC)
        if scaler is not None:
            scaler.scale(loss).backward()
            if max_grad_norm is not None:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(opt); scaler.update()
        else:
            loss.backward()
            if max_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            opt.step()
        total += loss.item() * g.num_nodes; nodes += g.num_nodes
    return total / max(nodes, 1)

@torch.no_grad()
def eval_epoch(model, loader, dyn_features_idx_y, y_scaler, device='cuda', lam_smooth=1e-3):
    model.eval()
    total, nodes = 0.0, 0
    for g in tqdm(loader, leave=False):
        g = g.to(device)
        pred = model(g.x, g.edge_index, getattr(g, 'edge_attr', None))
        bc_mask = getattr(g, 'bc_mask', None)
        if bc_mask is not None:
          bc_mask = bc_mask.bool()
          mask_free = ~bc_mask
        else:
          mask_free = None
        rigid_mask = getattr(g, 'rigid_mask', None)
        loss = smooth_edge_penalty(pred, g.y, g.edge_index, lam_smooth, bc_mask, rigid_mask)
        loss += masked_mse(pred, g.y, mask_free)
        loss += loss_bc_zero_disp(pred, dyn_features_idx_y, bc_mask, y_scaler, LAM_BC)

        # Accumulators
        total += loss.item() * g.num_nodes
        nodes += g.num_nodes
    return total / max(nodes, 1)



In [20]:
def save_checkpoint(path, model, x_scaler, y_scaler, edge_scaler):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({"model_state": model.state_dict(),
                "x_scaler": x_scaler, "y_scaler": y_scaler, "edge_scaler": edge_scaler}, path)
    print("Saved best checkpoint ->", path)

In [ ]:
import time, torch, torch.optim as optim
from collections import defaultdict

start = time.time()
print("Building model...")

edge_dim = train_graphs[0].edge_attr.size(1) if hasattr(train_graphs[0], "edge_attr") and train_graphs[0].edge_attr is not None else 0
assert edge_dim > 0, "edge_attr required for GINEConv; si no tienes, cambia a un modelo sin edge_attr."

model = ImpactGNN_Edge(in_ch=INPUT_FEATURES, edge_attr_dim=edge_dim,
                       hidden=HIDDEN, out_ch=OUTPUT_FEATURES, layers=N_LAYERS, dropout=0.1).to(device)

opt = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scaler = torch.cuda.amp.GradScaler(enabled=(device=='cuda'))

checkpoint_path = "/content/best_impact_gnn.pt"

best_val = float('inf')
best_state = None
wait = 15

print(f"Training for up to {MAX_EPOCHS} epochs...")

for epoch in range(1, MAX_EPOCHS+1):
    tr = train_epoch(model, train_loader, opt, Y_DYNAMIC_INDEX, y_scaler, device='cuda', lam_smooth=LAM_SMOOTH, scaler=scaler, max_grad_norm=1.0)
    vl = eval_epoch(model, val_loader, Y_DYNAMIC_INDEX, y_scaler, device='cuda', lam_smooth=LAM_SMOOTH)
    print(f"[Epoch {epoch:03d}] train {tr:.6f} | val {vl:.6f}")

    if vl + 1e-6 < best_val:
        wait = 0
        best_val = vl
        best_state = {k: v.cpu() for k,v in model.state_dict().items()}
        save_checkpoint(checkpoint_path, model, x_scaler, y_scaler, edge_scaler)
    else:
        wait += 1
        if wait >= PATIENCE:
          print(f"Early stopping after {PATIENCE} epochs without improvement")
          break
end = time.time()
print(f"Elapsed time for training: {end - start:.1f}s.")

Building model...
Training for up to 200 epochs...


/tmp/ipython-input-568115549.py:14: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device=='cuda'))


  0%|          | 0/392 [00:00<?, ?it/s]

/tmp/ipython-input-1847835057.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(scaler is not None)):


  0%|          | 0/86 [00:00<?, ?it/s]

[Epoch 001] train 0.266523 | val 0.034522
Saved best checkpoint -> /content/best_impact_gnn.pt


  0%|          | 0/392 [00:00<?, ?it/s]

  0%|          | 0/86 [00:00<?, ?it/s]

[Epoch 002] train 0.066282 | val 0.023326
Saved best checkpoint -> /content/best_impact_gnn.pt


  0%|          | 0/392 [00:00<?, ?it/s]

# Rollout

In [ ]:
@torch.no_grad()
def rollout(
    model,
    T_eff: int,
    x0: torch.Tensor,                 # (N, Din) EN ESPACIO FÍSICO (desnormalizado)
    dyn_idx_x,                          # lista/LongTensor de índices dinámicos en x
    edge_index,
    edge_attr,
    x_scaler,                         # (x_mean, x_std) de Din cols
    y_scaler,                         # (y_mean, y_std) de Dout cols (las dinámicas que predices)
    bc_mask=None,                     # Bool [N] (opcional)
    clamp_bc=False,                   # si quieres forzar 0 en desplazamientos de nodos fijos
    device='cuda',
    return_full=False                 # True -> devuelve la secuencia de x_t completas; False -> sólo y_hat por paso
):
    """
    x0: estado inicial con TODAS las columnas de entrada del modelo (Din).
        Si alguna estática no la tienes en x0, añádela antes.
    dyn_idx: posiciones en x que el modelo predice y que se actualizan en cada paso.
    disp_idx_in_dyn_x: subset dentro de las dinámicas que corresponde a desplazamientos.
    """
    xm, xs = x_scaler
    ym, ys = y_scaler

    xm_d, xs_d = xm.to(device), xs.to(device)
    ym_d, ys_d = ym.to(device), ys.to(device)

    x_t = x0.to(device)                              # (N, Din)


    # print(f"Initial : x_t{x_t[:10,:]}")
    edge_index = edge_index.to(device)
    edge_attr  = edge_attr.to(device) if edge_attr is not None else None

    preds = []
    states = [x_t.clone()]

    for i in range(T_eff - 1):
        # vals_per_axis, idx_per_axis = x_t[:, :3].max(dim=0)  # -> (3,), (3,)
        # max_delta_x, max_delta_y, max_delta_z = vals_per_axis
        # print(f"Max desplazamiento input X: {max_delta_x}")
        # print(f"Max desplazamiento input Y: {max_delta_y}")
        # print(f"Max desplazamiento input Z: {max_delta_z}")


        # normalize the input
        x_in = (x_t - xm_d) / xs_d         # (N, Din)

        # normalized output
        y_pred_norm = model(x_in, edge_index, edge_attr)      # (N, Dout)
        # assert y_pred_norm.size(1) == dyn_idx_x.numel(), f"Dout={y_pred_norm.size(1)} debe coincidir con len(dyn_idx_x)={dyn_idx_x.numel()}"
        # desnormalize the output
        y_pred = y_pred_norm * ys_d + ym_d   # (N, Dout)

        # if i <= 2 or (i > 50 and i < 60):
        #   print(f"T{i} : x_t{x_t[:10,:]}")
        #   print(f"T{i} : y_t{y_pred[:10,:]}")

        # vals_per_axis, idx_per_axis = y_hat[:, :3].max(dim=0)  # -> (3,), (3,)
        # max_delta_x, max_delta_y, max_delta_z = vals_per_axis
        # print(f"Max desplazamiento output X: {max_delta_x}")
        # print(f"Max desplazamiento output Y: {max_delta_y}")
        # print(f"Max desplazamiento output Z: {max_delta_z}")

        # clamp a 0 en desplazamientos de nodos fijos (si aplica)
        # TODO: Implement
        if clamp_bc and bc_mask is not None and dyn_idx_x is not None:
            if isinstance(dyn_idx_x, (list, tuple)):
                dyn_idx_x = torch.as_tensor(dyn_idx_x, device=device)
            # y_hat[bc_mask, disp_idx_in_dyn] = 0
            # como y_hat es (N,Dout), indexa filas y columnas:
            y_hat_bc = y_pred[bc_mask]                        # (Nb, Dout)
            y_hat_bc[:, dyn_idx_x] = 0.0
            y_pred[bc_mask] = y_hat_bc

        # actualiza x_t SOLO en las columnas dinámicas
        x_t = x_t.clone()
        x_t[:, dyn_idx_x] = y_pred

        preds.append(y_pred)
        if return_full:
            states.append(x_t.clone())

    return (torch.stack(states, 0) if return_full else torch.stack(preds, 0))

# Evaluate results

In [ ]:
@torch.no_grad()
def compute_metrics(y_pred: torch.Tensor, y_real: torch.Tensor) -> Dict[str, float]:
    # Only compare the first 3 dimensions (displacements)
    diff = y_pred - y_real
    mae = (diff).abs().mean().item()
    rmse = torch.sqrt(((diff) ** 2).mean()).item()
    l2   = torch.norm(diff, dim=-1)  # [N]
    ade  = l2.mean().item()
    fde = (y_pred[-1] - y_real[-1]).abs().mean().item()
    return {'MAE': mae, 'RMSE': rmse, 'ADE': ade, 'FDE': fde}


In [ ]:
if best_state is not None: model.load_state_dict(best_state)

print("Evaluating rollout on TEST simulations…")
model.eval()
metrics_all = []
for sid, info in test_static.items():
    edge_index = info['edge_index']
    edge_attr  = transform_edge_attr(info.get('edge_attr', None), edge_scaler)
    T_eff = info['T_eff']
    y_real = info['y_real'].to(device)
    x0 = info['x0']
    bc_mask = info['bc_mask']

    pred = rollout(model, T_eff, x0, X_DYNAMIC_INDEX, edge_index, edge_attr, x_scaler, y_scaler, bc_mask, CLAMP_BC_IN_ROLLOUT, device, False)
    m = compute_metrics(pred, y_real)
    metrics_all.append(m)
    print(f"[SIM {sid}] MAE={m['MAE']:.6f} RMSE={m['RMSE']:.6f} ADE={m['ADE']:.6f} FDE={m['FDE']:.6f}")


if metrics_all:
    avg = {k: float(np.mean([d[k] for d in metrics_all])) for k in metrics_all[0].keys()}
    print("==== TEST AVERAGE ===="); [print(f"{k}: {v:.6f}") for k,v in avg.items()]

# Results analysis

# Report:

---
---
## 27/08/2025

### Experimento 1
Experimento sólo con las Δx, Δy y Δz de la iteración número 3 de la BBDD.

N_HIDDEN = 128, LAYERS = 3, MIN_T=5, LAM_BC = 1e-4, LAM_SMOOTH = 1e-4

Test Averages:

MAE: 21.562727

RMSE: 40.034506

ADE: 52.877555

FDE: 17.383746

---
### Experimento 2
Experimento sólo con las Δx, Δy y Δz de la iteración número 3 de la BBDD.

**Hemos cambiado la manera en la que se crea la x0 haciendo el detach porque sino al ser una referencia, se normaliza, teniendo una normalización constantemente. Aunque la prediccion explota al menos el rollout ya no se queda congelado.**

N_HIDDEN = 128, LAYERS = 3, MIN_T=5, LAM_BC = 1e-4, LAM_SMOOTH = 1e-4

Test Averages:

MAE: 11354192.828125

RMSE: 87994739.437500

ADE: 20427097.062500

FDE: 233210583.500000

---
### Experimento 3
Experimento sólo con las Δx, Δy y Δz de la iteración número 3 de la BBDD.

**Hemos cambiado incrementado el LAM_BC y el LAM_SMOOTH a 1e-3 ambos**

N_HIDDEN = 128, LAYERS = 3, MIN_T=5, LAM_BC = 1e-3, LAM_SMOOTH = 1e-3

Test Averages:

MAE: 54.106339

RMSE: 87.332501

ADE: 126.598473

FDE: 67.258650

---

3D Animation

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from IPython.display import HTML

def unique_undirected_edges(edge_index: torch.Tensor):
    ei = edge_index.detach().cpu().numpy().T
    undirected = set()
    for u, v in ei:
        if u == v: continue
        a, b = (u, v) if u < v else (v, u)
        undirected.add((a, b))
    return np.array(list(undirected), dtype=np.int64)

def make_line_collection(pos_xyz: np.ndarray, edges_uv: np.ndarray, color='k', alpha=0.4, linewidth=0.6):
    segs = np.stack([pos_xyz[edges_uv[:,0]], pos_xyz[edges_uv[:,1]]], axis=1)
    return Line3DCollection(segs, linewidths=linewidth, colors=color, alpha=alpha)

def update_line_collection(lc: Line3DCollection, pos_xyz: np.ndarray, edges_uv: np.ndarray):
    segs = np.stack([pos_xyz[edges_uv[:,0]], pos_xyz[edges_uv[:,1]]], axis=1)
    lc.set_segments(segs)

def animate_simulation(sim_info: dict, model, x_scaler, y_scaler, dyn_idx_x, edge_scaler=None, device='cuda',
                       save_path="/content/rollout.mp4", max_edges=4000, elev=25, azim=40, show_inline=True):
    pos0       = sim_info['pos0']
    edge_index = sim_info['edge_index']
    edge_attr  = transform_edge_attr(sim_info.get('edge_attr', None), edge_scaler)  # 👈 escalar
    T_eff      = sim_info['T_eff']
    y_real    = sim_info['y_real']   # (T_eff-1,N,N_Attributes)
    x0    = sim_info['x0']

    # --- BC mask (opcional) ---
    bc_mask = sim_info.get('bc_mask', None)
    rigid_mask = sim_info.get('rigid_mask', None)
    fixed_idx = sim_info.get('fixed_idx', None)
    if bc_mask is None and fixed_idx is not None:
        # construye máscara a partir de índices si es lo que guardas
        N = y_real.shape[1]
        bc_mask = torch.zeros(N, dtype=torch.bool)
        bc_mask[fixed_idx] = True

    model.eval()
    with torch.no_grad():
        pred = rollout(model, T_eff, x0, dyn_idx_x, edge_index, edge_attr, x_scaler, y_scaler, bc_mask, CLAMP_BC_IN_ROLLOUT, device, False)

    assert pos0 is not None
    pos0_np = (pos0).detach().cpu().numpy()
    gt_np   = y_real.detach().cpu().numpy()
    gt_np = gt_np[..., :3] # We only want the displacements
    pr_np   = pred.detach().cpu().numpy()
    pr_np = pr_np[..., :3] # We only want the displacements
    Tm1, N, _ = gt_np.shape # Number of timesteps and nodes

    edges_uv = unique_undirected_edges(edge_index)
    if max_edges is not None and len(edges_uv) > max_edges:
        idx = np.random.RandomState(0).choice(len(edges_uv), size=max_edges, replace=False)
        edges_uv = edges_uv[idx]

    all_gt = pos0_np[None,...] + gt_np
    all_pr = pos0_np[None,...] + pr_np
    # xyz_min = np.minimum(all_gt.min(axis=(0,1)), all_pr.min(axis=(0,1)))
    xyz_min = all_gt.min(axis=(0,1))
    # xyz_max = np.maximum(all_gt.max(axis=(0,1)), all_pr.max(axis=(0,1)))
    xyz_max = all_gt.max(axis=(0,1))
    pad = 0.05 * (xyz_max - xyz_min + 1e-9)
    xyz_min -= pad; xyz_max += pad

    fig = plt.figure(figsize=(12,6))
    ax_gt   = fig.add_subplot(121, projection='3d')
    ax_pr   = fig.add_subplot(122, projection='3d')
    for ax, title in [(ax_gt, "Ground Truth"), (ax_pr, "Prediction")]:
        ax.set_xlim([xyz_min[0], xyz_max[0]]); ax.set_ylim([xyz_min[1], xyz_max[1]]); ax.set_zlim([xyz_min[2], xyz_max[2]])
        ax.view_init(elev=elev, azim=azim); ax.set_title(title)

    pos_gt0 = pos0_np + gt_np[0]
    pos_pr0 = pos0_np + pr_np[0]
    lc_gt = make_line_collection(pos_gt0, edges_uv, color='tab:green', alpha=0.6, linewidth=0.7)
    lc_pr = make_line_collection(pos_pr0, edges_uv, color='tab:red',   alpha=0.6, linewidth=0.7)
    ax_gt.add_collection3d(lc_gt); ax_pr.add_collection3d(lc_pr)

    # --- Scatter con o sin BC ---
    use_bc = bc_mask is not None
    if use_bc:
        bc_np   = bc_mask.detach().cpu().numpy().astype(bool)
        free_np = ~bc_np
        rigid_np   = rigid_mask.detach().cpu().numpy().astype(bool)
        no_rigid_np = ~rigid_np

        # GT
        sc_gt_free = ax_gt.scatter(pos_gt0[free_np,0], pos_gt0[free_np,1], pos_gt0[free_np,2],
                                   s=4, c='tab:green', alpha=0.85, marker='o', label='free')
        sc_gt_fix  = ax_gt.scatter(pos_gt0[bc_np,0],   pos_gt0[bc_np,1],   pos_gt0[bc_np,2],
                                   s=14, c='gold',     alpha=0.95, marker='^', label='BC fixed')

        # Pred
        sc_pr_free = ax_pr.scatter(pos_pr0[free_np,0], pos_pr0[free_np,1], pos_pr0[free_np,2],
                                   s=4, c='tab:gren',   alpha=0.85, marker='o', label='free')
        sc_pr_fix  = ax_pr.scatter(pos_pr0[bc_np,0],   pos_pr0[bc_np,1],   pos_pr0[bc_np,2],
                                   s=14, c='gold', alpha=0.95, marker='^', label='BC fixed')

        # leyenda compacta
        ax_gt.legend(loc='upper left', fontsize=8, frameon=False)
        ax_pr.legend(loc='upper left', fontsize=8, frameon=False)
    else:
        sc_gt = ax_gt.scatter(pos_gt0[:,0], pos_gt0[:,1], pos_gt0[:,2], s=2, c='tab:green', alpha=0.8)
        sc_pr = ax_pr.scatter(pos_pr0[:,0], pos_pr0[:,1], pos_pr0[:,2], s=2, c='tab:red',   alpha=0.8)

    def update(frame):
        pos_gt = pos0_np + gt_np[frame]
        pos_pr = pos0_np + pr_np[frame]

        update_line_collection(lc_gt, pos_gt, edges_uv)
        update_line_collection(lc_pr, pos_pr, edges_uv)

        if use_bc:
            # actualiza offsets 3D para cada grupo
            bc_np   = bc_mask.detach().cpu().numpy().astype(bool)
            free_np = ~bc_np

            sc_gt_free._offsets3d = (pos_gt[free_np,0], pos_gt[free_np,1], pos_gt[free_np,2])
            sc_gt_fix._offsets3d  = (pos_gt[bc_np,0],   pos_gt[bc_np,1],   pos_gt[bc_np,2])
            sc_pr_free._offsets3d = (pos_pr[free_np,0], pos_pr[free_np,1], pos_pr[free_np,2])
            sc_pr_fix._offsets3d  = (pos_pr[bc_np,0],   pos_pr[bc_np,1],   pos_pr[bc_np,2])
            artists = (lc_gt, lc_pr, sc_gt_free, sc_gt_fix, sc_pr_free, sc_pr_fix)
        else:
            sc_gt._offsets3d = (pos_gt[:,0], pos_gt[:,1], pos_gt[:,2])
            sc_pr._offsets3d = (pos_pr[:,0], pos_pr[:,1], pos_pr[:,2])
            artists = (lc_gt, lc_pr, sc_gt, sc_pr)

        ax_gt.set_title(f"Ground Truth – step {frame+1}/{Tm1}")
        ax_pr.set_title(f"Prediction – step {frame+1}/{Tm1}")
        return artists

    ani = animation.FuncAnimation(fig, update, frames=Tm1, interval=120, blit=False)
    try:
        ani.save(save_path, writer=animation.FFMpegWriter(fps=8, bitrate=2000))
        print("Saved animation to:", save_path)
    except Exception as e:
        print("FFMpeg failed:", e)
    plt.close(fig)
    if show_inline:
        display(HTML(ani.to_jshtml()))
    return ani



## Creating animation

In [ ]:

print("Creating  test animation...")
sid = next(iter(test_static.keys()))
animation_name = "rollout_test_" + str(sid) + ".mp4"
animation_path = "/content/drive/MyDrive/CrashGeoNN/" + animation_name
_ = animate_simulation(test_static[sid], model, x_scaler, y_scaler, X_DYNAMIC_INDEX, edge_scaler=edge_scaler, device='cuda',
                       save_path=animation_path, max_edges=4000, elev=25, azim=40, show_inline=False)

# print("Creating  train animation...")
# sid = next(iter(train_static.keys()))
# animation_name = "rollout_train_" + str(sid) + ".mp4"
# animation_path = "/content/drive/MyDrive/CrashGeoNN/" + animation_name
# _ = animate_simulation(test_static[sid], model, x_scaler, y_scaler, X_DYNAMIC_INDEX, edge_scaler=edge_scaler, device='cuda',
#                        save_path=animation_path, max_edges=4000, elev=25, azim=40, show_inline=False)

print("Finished!")
